# Load data

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)

train = pd.read_parquet(
    "../data/processed/train.parquet"
)

validation = pd.read_parquet(
    "../data/processed/validation.parquet"
)

print("Train:", train.shape)
print("Validation:", validation.shape)

PyTorch: 2.6.0+cpu
Train: (1928949, 5)
Validation: (413345, 5)


# Aggregate interactions

In [2]:
user_item = (
    train
    .groupby(["user_id", "item_id"])["interaction_strength"]
    .sum()
    .reset_index()
)

print(user_item.shape)
print(user_item.head())

(1496539, 3)
   user_id  item_id  interaction_strength
0        3   385090                     1
1        5    61396                     1
2        7   139394                     1
3        7   164941                     1
4        7   226353                     1


# Create compact IDs

In [3]:
user_ids = user_item["user_id"].unique()
item_ids = user_item["item_id"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

item_to_idx = {
    item_id: idx
    for idx, item_id in enumerate(item_ids)
}

idx_to_user = {
    idx: user_id
    for user_id, idx in user_to_idx.items()
}

idx_to_item = {
    idx: item_id
    for item_id, idx in item_to_idx.items()
}

print("Users:", len(user_ids))
print("Items:", len(item_ids))

Users: 978906
Items: 200974


# Build positive interaction sets

In [4]:
user_positive_items = {}

for row in user_item.itertuples(index=False):

    user_idx = user_to_idx[row.user_id]
    item_idx = item_to_idx[row.item_id]

    if user_idx not in user_positive_items:
        user_positive_items[user_idx] = set()

    user_positive_items[user_idx].add(item_idx)

print(
    "Users with interactions:",
    len(user_positive_items)
)

Users with interactions: 978906


# BPR Dataset

In [5]:
class BPRDataset(Dataset):

    def __init__(
        self,
        user_positive_items,
        num_items,
        num_samples=None
    ):

        self.user_positive_items = (
            user_positive_items
        )

        self.users = list(
            user_positive_items.keys()
        )

        self.num_items = num_items

        if num_samples is None:
            num_samples = len(self.users) * 5

        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):

        user = np.random.choice(
            self.users
        )

        positive = np.random.choice(
            list(
                self.user_positive_items[user]
            )
        )

        while True:

            negative = np.random.randint(
                0,
                self.num_items
            )

            if negative not in (
                self.user_positive_items[user]
            ):
                break

        return (
            user,
            positive,
            negative
        )

# Start with a reasonable training size

In [6]:
dataset = BPRDataset(
    user_positive_items,
    num_items=len(item_ids),
    num_samples=2_000_000
)

print("Training samples:", len(dataset))

Training samples: 2000000


In [7]:
dataloader = DataLoader(
    dataset,
    batch_size=2048,
    shuffle=False,
    num_workers=0
)

# Define the BPR model

In [8]:
class BPRMatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=64
    ):

        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        self.user_bias = nn.Embedding(
            num_users,
            1
        )

        self.item_bias = nn.Embedding(
            num_items,
            1
        )

        self._initialize()

    def _initialize(self):

        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.item_embedding.weight,
            std=0.01
        )

        nn.init.zeros_(
            self.user_bias.weight
        )

        nn.init.zeros_(
            self.item_bias.weight
        )

    def forward(
        self,
        users,
        positive_items,
        negative_items
    ):

        user_vec = self.user_embedding(
            users
        )

        pos_vec = self.item_embedding(
            positive_items
        )

        neg_vec = self.item_embedding(
            negative_items
        )

        pos_score = (
            (user_vec * pos_vec).sum(dim=1)
            + self.user_bias(users).squeeze()
            + self.item_bias(
                positive_items
            ).squeeze()
        )

        neg_score = (
            (user_vec * neg_vec).sum(dim=1)
            + self.user_bias(users).squeeze()
            + self.item_bias(
                negative_items
            ).squeeze()
        )

        return pos_score, neg_score

# Device

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


# Create model

In [10]:
model = BPRMatrixFactorization(
    num_users=len(user_ids),
    num_items=len(item_ids),
    embedding_dim=64
).to(device)

print(model)

BPRMatrixFactorization(
  (user_embedding): Embedding(978906, 64)
  (item_embedding): Embedding(200974, 64)
  (user_bias): Embedding(978906, 1)
  (item_bias): Embedding(200974, 1)
)


# Optimizer

In [11]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-6
)

# BPR loss

In [12]:
def bpr_loss(
    positive_scores,
    negative_scores
):

    difference = (
        positive_scores -
        negative_scores
    )

    return -torch.log(
        torch.sigmoid(difference) + 1e-8
    ).mean()

# Train

In [13]:
class BPRMatrixFactorization(nn.Module):

    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=32
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim,
            sparse=True
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim,
            sparse=True
        )

        self._initialize()

    def _initialize(self):

        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.item_embedding.weight,
            std=0.01
        )

    def forward(
        self,
        users,
        positive_items,
        negative_items
    ):

        user_vec = self.user_embedding(users)

        pos_vec = self.item_embedding(
            positive_items
        )

        neg_vec = self.item_embedding(
            negative_items
        )

        pos_score = (
            user_vec * pos_vec
        ).sum(dim=1)

        neg_score = (
            user_vec * neg_vec
        ).sum(dim=1)

        return pos_score, neg_score

In [14]:
model = BPRMatrixFactorization(
    num_users=len(user_ids),
    num_items=len(item_ids),
    embedding_dim=32
).to(device)

optimizer = torch.optim.SparseAdam(
    model.parameters(),
    lr=0.001
)

In [15]:
NUM_SAMPLES = 500_000

In [16]:
dataset = BPRDataset(
    user_positive_items,
    num_items=len(item_ids),
    num_samples=NUM_SAMPLES
)

In [17]:
dataloader = DataLoader(
    dataset,
    batch_size=8192,
    shuffle=False,
    num_workers=0
)

In [19]:
# Minimum activity thresholds
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

user_counts = (
    user_item
    .groupby("user_id")
    .size()
)

item_counts = (
    user_item
    .groupby("item_id")
    .size()
)

active_users = user_counts[
    user_counts >= MIN_USER_INTERACTIONS
].index

active_items = item_counts[
    item_counts >= MIN_ITEM_INTERACTIONS
].index

cf_data = user_item[
    user_item["user_id"].isin(active_users)
    &
    user_item["item_id"].isin(active_items)
].copy()

print("Original interactions:", len(user_item))
print("Active interactions:", len(cf_data))
print("Active users:", cf_data["user_id"].nunique())
print("Active items:", cf_data["item_id"].nunique())

Original interactions: 1496539
Active interactions: 262661
Active users: 27771
Active items: 42672


In [20]:
cf_user_ids = cf_data["user_id"].unique()
cf_item_ids = cf_data["item_id"].unique()

cf_user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(cf_user_ids)
}

cf_item_to_idx = {
    item_id: idx
    for idx, item_id in enumerate(cf_item_ids)
}

cf_data["user_idx"] = (
    cf_data["user_id"]
    .map(cf_user_to_idx)
)

cf_data["item_idx"] = (
    cf_data["item_id"]
    .map(cf_item_to_idx)
)

In [21]:
user_positive = (
    cf_data
    .groupby("user_idx")["item_idx"]
    .apply(np.array)
)

print(
    "Active users:",
    len(user_positive)
)

Active users: 27771


In [23]:
def sample_bpr_batch(
    batch_size,
    user_positive,
    num_items
):
    
    users = np.random.randint(
        0,
        len(user_positive),
        size=batch_size
    )

    positives = np.empty(
        batch_size,
        dtype=np.int64
    )

    negatives = np.random.randint(
        0,
        num_items,
        size=batch_size
    )

    for i, user in enumerate(users):
        
        positives[i] = np.random.choice(
            user_positive[user]
        )

    # Retry invalid negatives
    for i in range(batch_size):
        
        while negatives[i] in user_positive[users[i]]:
            
            negatives[i] = np.random.randint(
                0,
                num_items
            )

    return users, positives, negatives

In [24]:
NUM_FACTORS = 32

In [25]:
model = BPRMatrixFactorization(
    num_users=len(cf_user_ids),
    num_items=len(cf_item_ids),
    embedding_dim=NUM_FACTORS
).to(device)

In [26]:
optimizer = torch.optim.SparseAdam(
    model.parameters(),
    lr=0.002
)

In [27]:
BATCH_SIZE = 4096
STEPS_PER_EPOCH = 100
EPOCHS = 2

In [28]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    for step in range(STEPS_PER_EPOCH):

        users, positives, negatives = (
            sample_bpr_batch(
                BATCH_SIZE,
                user_positive,
                len(cf_item_ids)
            )
        )

        users = torch.tensor(
            users,
            dtype=torch.long,
            device=device
        )

        positives = torch.tensor(
            positives,
            dtype=torch.long,
            device=device
        )

        negatives = torch.tensor(
            negatives,
            dtype=torch.long,
            device=device
        )

        optimizer.zero_grad()

        pos_scores, neg_scores = model(
            users,
            positives,
            negatives
        )

        loss = bpr_loss(
            pos_scores,
            neg_scores
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Loss: "
        f"{total_loss / STEPS_PER_EPOCH:.6f}"
    )

Epoch 1/2 - Loss: 0.692118
Epoch 2/2 - Loss: 0.678281


In [29]:
print("Active users:", len(cf_user_ids))
print("Active items:", len(cf_item_ids))
print("Active interactions:", len(cf_data))

print("\nInteractions per user:")
print(
    cf_data.groupby("user_id")
    .size()
    .describe()
)

print("\nInteractions per item:")
print(
    cf_data.groupby("item_id")
    .size()
    .describe()
)

Active users: 27771
Active items: 42672
Active interactions: 262661

Interactions per user:
count    27771.000000
mean         9.458104
std         32.704418
min          1.000000
25%          5.000000
50%          6.000000
75%          8.000000
max       2413.000000
dtype: float64

Interactions per item:
count    42672.000000
mean         6.155348
std          9.087849
min          1.000000
25%          2.000000
50%          3.000000
75%          7.000000
max        286.000000
dtype: float64


In [30]:
print(
    "Users in positive lookup:",
    len(user_positive)
)

print(
    "Example:",
    user_positive.iloc[0][:10]
)

Users in positive lookup: 27771
Example: [0 1 2 3 4 5]


In [31]:
NUM_FACTORS = 32

model = BPRMatrixFactorization(
    num_users=len(cf_user_ids),
    num_items=len(cf_item_ids),
    embedding_dim=NUM_FACTORS
).to(device)

optimizer = torch.optim.SparseAdam(
    model.parameters(),
    lr=0.002
)

print(model)

BPRMatrixFactorization(
  (user_embedding): Embedding(27771, 32, sparse=True)
  (item_embedding): Embedding(42672, 32, sparse=True)
)


In [32]:
BATCH_SIZE = 4096
STEPS_PER_EPOCH = 100
EPOCHS = 2

In [33]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    for step in range(STEPS_PER_EPOCH):

        users, positives, negatives = sample_bpr_batch(
            BATCH_SIZE,
            user_positive,
            len(cf_item_ids)
        )

        users = torch.tensor(
            users,
            dtype=torch.long,
            device=device
        )

        positives = torch.tensor(
            positives,
            dtype=torch.long,
            device=device
        )

        negatives = torch.tensor(
            negatives,
            dtype=torch.long,
            device=device
        )

        optimizer.zero_grad()

        pos_scores, neg_scores = model(
            users,
            positives,
            negatives
        )

        loss = bpr_loss(
            pos_scores,
            neg_scores
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = (
        total_loss / STEPS_PER_EPOCH
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Loss: {avg_loss:.6f}"
    )

Epoch 1/2 - Loss: 0.692141
Epoch 2/2 - Loss: 0.678700


## Test BPR Recommendations

# Create the reverse mappings

In [34]:
idx_to_cf_user = {
    idx: user_id
    for user_id, idx in cf_user_to_idx.items()
}

idx_to_cf_item = {
    idx: item_id
    for item_id, idx in cf_item_to_idx.items()
}

# Recommendation function

In [35]:
def recommend_bpr(user_id, k=10):

    if user_id not in cf_user_to_idx:
        return pd.DataFrame(
            columns=["item_id", "score"]
        )

    user_idx = cf_user_to_idx[user_id]

    user_tensor = torch.tensor(
        [user_idx],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        user_vector = (
            model.user_embedding(
                user_tensor
            )
            .cpu()
            .numpy()[0]
        )

        item_vectors = (
            model.item_embedding.weight
            .detach()
            .cpu()
            .numpy()
        )

    scores = item_vectors @ user_vector

    # Remove training-seen items
    seen_items = set(
        user_positive[user_idx]
    )

    for item_idx in seen_items:
        scores[item_idx] = -np.inf

    top_indices = np.argpartition(
        scores,
        -k
    )[-k:]

    top_indices = top_indices[
        np.argsort(
            scores[top_indices]
        )[::-1]
    ]

    return pd.DataFrame({
        "item_id": [
            idx_to_cf_item[idx]
            for idx in top_indices
        ],
        "score": scores[top_indices]
    })

# Test one user

In [36]:
sample_user = cf_user_ids[0]

print(
    "Sample user:",
    sample_user
)

print(
    recommend_bpr(
        sample_user,
        k=10
    )
)

Sample user: 64
   item_id     score
0    15189  0.087944
1   199018  0.087436
2   312717  0.082957
3   315543  0.080079
4   266900  0.079041
5   415825  0.077776
6   159822  0.077430
7   240900  0.076472
8    20740  0.069343
9   159095  0.068545


# Check that recommendations aren't seen items

In [37]:
sample_user_idx = cf_user_to_idx[
    sample_user
]

seen_item_ids = {
    idx_to_cf_item[idx]
    for idx in user_positive[
        sample_user_idx
    ]
}

recommendations = recommend_bpr(
    sample_user,
    k=10
)

print(
    "Recommended items:",
    set(recommendations["item_id"])
)

print(
    "Overlap with training:",
    set(recommendations["item_id"])
    & seen_item_ids
)

Recommended items: {240900, 20740, 199018, 312717, 159822, 415825, 266900, 15189, 315543, 159095}
Overlap with training: set()


# Find eligible validation users

In [38]:
active_validation_users = (
    set(validation["user_id"].unique())
    & set(cf_user_ids)
)

print(
    "BPR validation users:",
    len(active_validation_users)
)

BPR validation users: 2503


In [39]:
active_validation = validation[
    validation["user_id"].isin(
        active_validation_users
    )
    &
    validation["item_id"].isin(
        cf_item_ids
    )
].copy()

print(
    "Validation interactions on known BPR items:",
    len(active_validation)
)

print(
    "Validation users with known items:",
    active_validation["user_id"].nunique()
)

Validation interactions on known BPR items: 18981
Validation users with known items: 2047


# Build BPR ground truth

In [40]:
bpr_ground_truth = (
    active_validation
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

bpr_eval_users = np.array(
    sorted(
        set(bpr_ground_truth.keys())
        & set(cf_user_ids)
    )
)

print(
    "Final BPR evaluation users:",
    len(bpr_eval_users)
)

Final BPR evaluation users: 2047


# Build BPR evaluation function

In [41]:
def evaluate_bpr(
    user_ids_eval,
    ks=(5, 10, 20)
):

    results = {
        k: {
            "precision": [],
            "recall": [],
            "ndcg": [],
            "hit": []
        }
        for k in ks
    }

    for count, user_id in enumerate(
        user_ids_eval,
        start=1
    ):

        recommendations = recommend_bpr(
            user_id,
            k=max(ks)
        )

        recommended = (
            recommendations["item_id"]
            .tolist()
        )

        relevant = bpr_ground_truth[
            user_id
        ]

        for k in ks:

            results[k]["precision"].append(
                precision_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

            results[k]["recall"].append(
                recall_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

            results[k]["ndcg"].append(
                ndcg_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

            results[k]["hit"].append(
                hit_rate_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

        if count % 250 == 0:
            print(
                f"Evaluated {count}/"
                f"{len(user_ids_eval)} users"
            )

    output = []

    for k in ks:

        output.append({
            "Model": "BPR",
            "K": k,
            "Precision@K": np.mean(
                results[k]["precision"]
            ),
            "Recall@K": np.mean(
                results[k]["recall"]
            ),
            "NDCG@K": np.mean(
                results[k]["ndcg"]
            ),
            "HitRate@K": np.mean(
                results[k]["hit"]
            )
        })

    return pd.DataFrame(output)

In [42]:
import numpy as np
import pandas as pd


def precision_at_k(recommended, relevant, k):
    if k == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / k


def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / len(relevant)


def hit_rate_at_k(recommended, relevant, k):
    return float(
        any(
            item in relevant
            for item in recommended[:k]
        )
    )


def ndcg_at_k(recommended, relevant, k):

    if len(relevant) == 0:
        return 0.0

    dcg = 0.0

    for rank, item in enumerate(
        recommended[:k],
        start=1
    ):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(
        len(relevant),
        k
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [43]:
bpr_results = evaluate_bpr(
    bpr_eval_users,
    ks=(5, 10, 20)
)

print(bpr_results)

Evaluated 250/2047 users
Evaluated 500/2047 users
Evaluated 750/2047 users
Evaluated 1000/2047 users
Evaluated 1250/2047 users
Evaluated 1500/2047 users
Evaluated 1750/2047 users
Evaluated 2000/2047 users
  Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0   BPR   5     0.004690  0.004576  0.005386   0.018564
1   BPR  10     0.003371  0.006913  0.005625   0.023449
2   BPR  20     0.002540  0.010508  0.006703   0.034685


In [44]:
np.save(
    "../data/processed/bpr_eval_users.npy",
    bpr_eval_users
)

print(
    "Saved:",
    len(bpr_eval_users)
)

Saved: 2047


In [45]:
import torch
import os

os.makedirs("../models", exist_ok=True)

torch.save(
    {
        "user_embedding":
            model.user_embedding.weight.detach().cpu(),

        "item_embedding":
            model.item_embedding.weight.detach().cpu(),

        "cf_user_to_idx":
            cf_user_to_idx,

        "cf_item_to_idx":
            cf_item_to_idx
    },
    "../models/bpr_model.pt"
)

print("BPR model saved.")

print(
    "User embeddings:",
    model.user_embedding.weight.shape
)

print(
    "Item embeddings:",
    model.item_embedding.weight.shape
)

BPR model saved.
User embeddings: torch.Size([27771, 32])
Item embeddings: torch.Size([42672, 32])


In [46]:
print("Training columns:")
print(train.columns.tolist())

print("\nTraining shape:")
print(train.shape)

print("\nUnique users:")
print(train["user_id"].nunique())

print("\nUnique items:")
print(train["item_id"].nunique())

Training columns:
['user_id', 'item_id', 'event', 'timestamp', 'interaction_strength']

Training shape:
(1928949, 5)

Unique users:
978906

Unique items:
200974


In [47]:
import pandas as pd

train = pd.read_parquet(
    "../data/processed/train.parquet"
)

print("Train:", train.shape)

user_counts = train["user_id"].value_counts()
item_counts = train["item_id"].value_counts()

for threshold in range(1, 21):

    active_users = (
        user_counts[
            user_counts >= threshold
        ].index
    )

    active_items = (
        item_counts[
            item_counts >= threshold
        ].index
    )

    active = train[
        train["user_id"].isin(active_users)
        &
        train["item_id"].isin(active_items)
    ]

    print(
        f"Threshold {threshold}: "
        f"users={active['user_id'].nunique()}, "
        f"items={active['item_id'].nunique()}, "
        f"interactions={len(active)}"
    )

Train: (1928949, 5)
Threshold 1: users=978906, items=200974, interactions=1928949
Threshold 2: users=280508, items=112283, interactions=1205993
Threshold 3: users=139162, items=79509, interactions=912564
Threshold 4: users=84247, items=62663, interactions=741340
Threshold 5: users=57475, items=52421, interactions=629426
Threshold 6: users=41457, items=45249, interactions=545719
Threshold 7: users=31639, items=40106, interactions=483218
Threshold 8: users=24970, items=36015, interactions=433437
Threshold 9: users=20144, items=32720, interactions=392405
Threshold 10: users=16571, items=29911, interactions=357718
Threshold 11: users=13965, items=27608, interactions=329416
Threshold 12: users=11917, items=25650, interactions=304748
Threshold 13: users=10332, items=23906, interactions=283838
Threshold 14: users=9007, items=22457, interactions=265198
Threshold 15: users=7959, items=21086, interactions=248707
Threshold 16: users=7070, items=19854, interactions=234176
Threshold 17: users=6334,